# Customer Churn Prediction: Profit-Based Evaluation of ML Models
## Step 6: Traditional Evaluation — Accuracy, Precision, Recall, F1, AUC

In [ ]:
# ── IMPORTS ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

sns.set_theme(style='whitegrid')

# Load data
X_test  = pd.read_csv('X_test.csv')
y_test  = pd.read_csv('y_test.csv').squeeze()

# Load trained models
models = {
    'Logistic Regression' : joblib.load('logistic_regression_model.pkl'),
    'Random Forest'       : joblib.load('random_forest_model.pkl'),
    'Gradient Boosting'   : joblib.load('gradient_boosting_model.pkl')
}

print('Data and models loaded ✓')

### 6.1 — Generate Predictions

In [ ]:
preds       = {}   # binary predictions  (0 or 1)
proba_preds = {}   # probability scores  (0.0 – 1.0)

for name, model in models.items():
    preds[name]       = model.predict(X_test)
    proba_preds[name] = model.predict_proba(X_test)[:, 1]  # probability of churn
    print(f'{name} predictions generated ✓')

### 6.2 — Metrics Table

In [ ]:
results = []

for name in models:
    y_pred  = preds[name]
    y_proba = proba_preds[name]
    results.append({
        'Model'     : name,
        'Accuracy'  : round(accuracy_score(y_test, y_pred)  * 100, 2),
        'Precision' : round(precision_score(y_test, y_pred) * 100, 2),
        'Recall'    : round(recall_score(y_test, y_pred)    * 100, 2),
        'F1 Score'  : round(f1_score(y_test, y_pred)        * 100, 2),
        'ROC-AUC'   : round(roc_auc_score(y_test, y_proba)  * 100, 2),
    })

results_df = pd.DataFrame(results).set_index('Model')

print('=== Traditional Evaluation Metrics (Test Set) ===')
print(results_df.to_string())

# Highlight best in each column
print('\n--- Best Model per Metric ---')
for col in results_df.columns:
    best = results_df[col].idxmax()
    print(f'  {col:<12}: {best} ({results_df.loc[best, col]}%)')

### 6.3 — Metrics Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

x      = np.arange(len(results_df.columns))
width  = 0.25
colors = ['#2E86AB', '#E74C3C', '#2ECC71']

for i, (model_name, color) in enumerate(zip(results_df.index, colors)):
    vals = results_df.loc[model_name].values
    bars = ax.bar(x + i * width, vals, width, label=model_name,
                  color=color, edgecolor='white', alpha=0.9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_title('Traditional Evaluation Metrics — All Models (Test Set)',
             fontsize=13, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(results_df.columns, fontsize=11)
ax.set_ylabel('Score (%)')
ax.set_ylim(50, 105)
ax.legend(fontsize=10)
ax.axhline(80, color='gray', linestyle='--', linewidth=1, alpha=0.5)

plt.tight_layout()
plt.savefig('plot_13_traditional_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved ✓')

### 6.4 — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, y_pred) in zip(axes, preds.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Predicted\nNo Churn', 'Predicted\nChurn'],
                yticklabels=['Actual\nNo Churn', 'Actual\nChurn'],
                linewidths=0.5, linecolor='white',
                annot_kws={'size': 14, 'weight': 'bold'})

    tn, fp, fn, tp = cm.ravel()
    ax.set_title(f'{name}\nTP={tp}  FP={fp}  FN={fn}  TN={tn}',
                 fontweight='bold', fontsize=11)

plt.suptitle('Confusion Matrices — All Models (Test Set)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_14_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key terms for presentation:')
print('  TP (True Positive)  = Correctly predicted churner → acted on, customer retained')
print('  FN (False Negative) = Missed churner → no action taken → customer lost (costly!)')
print('  FP (False Positive) = Non-churner flagged → unnecessary retention cost')
print('  TN (True Negative)  = Correctly identified loyal customer')

### 6.5 — ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#2E86AB', '#E74C3C', '#2ECC71']

for (name, y_proba), color in zip(proba_preds.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})',
            color=color, linewidth=2.5)

# Random baseline
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier (AUC = 0.500)')

ax.set_title('ROC Curves — All Models', fontsize=13, fontweight='bold')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig('plot_15_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved ✓')

### 6.6 — Model Ranking by Traditional Metrics

In [ ]:
# Rank models by each metric — this ranking will be compared to profit ranking in Step 7
print('=== MODEL RANKINGS — Traditional Metrics ===')
print('(Rank 1 = Best)\n')

ranking_df = results_df.rank(ascending=False, method='min').astype(int)
print(ranking_df.to_string())

# Overall rank by average rank across all metrics
ranking_df['Avg Rank'] = ranking_df.mean(axis=1).round(2)
ranking_df = ranking_df.sort_values('Avg Rank')

print('\n--- Overall Traditional Ranking ---')
for i, (model, row) in enumerate(ranking_df.iterrows(), 1):
    print(f'  #{i} {model}  (Avg Rank: {row["Avg Rank"]})')

# Save for comparison in Step 7
results_df.to_csv('traditional_metrics.csv')
ranking_df.to_csv('traditional_rankings.csv')
print('\nSaved: traditional_metrics.csv & traditional_rankings.csv ✓')

print('\n' + '='*55)
print('Step 6 Complete ✓  →  Proceed to Step 7: Profit-Based Evaluation')
print('='*55)
print('Note: The top-ranked model here may NOT be the most profitable.')
print('That is the core finding we will demonstrate in Step 7.')